# Word2Vec (Skip-Gram + negative sampling) на игрушечном корпусе

Перед каждой ячейкой с кодом — «🔮 Предскажи». Сначала пишешь предсказание в чат, оно переносится в markdown-ячейку, потом запуск.

In [ ]:
import torch
import matplotlib.pyplot as plt

from word2vec import (
    FUNCTION_WORDS, RARE, SGNS, TOPICS, build_vocab, make_toy_corpus, most_similar,
    negative_distribution, sample_negatives, skipgram_pairs, train,
)

sentences = make_toy_corpus(seed=0)
word2id, id2word, counts = build_vocab(sentences)
print(len(sentences), "предложений, пример:", sentences[:3])

## 1. Корпус и словарь

🔮 **Предскажи:** V; 10 самых частых слов; редкое слово каждой темы и его count; число пар Skip-Gram при `window=2`.

**Предсказание:** …

In [ ]:
V = len(id2word)
print("V =", V)
top = torch.argsort(counts, descending=True)[:10]
print("топ-10:", [(id2word[i], counts[i].item()) for i in top.tolist()])
for topic, w in RARE.items():
    print(f"редкое ({topic}): {w} — {counts[word2id[w]].item()}")

pairs = torch.tensor([p for s in sentences for p in skipgram_pairs([word2id[w] for w in s], window=2)])
print("пар при window=2:", len(pairs))

## 2. Распределение негативов

🔮 **Предскажи:** для 5 самых частых и 3 редких слов — у кого доля в $P_{neg} \propto f^{0.75}$ вырастет относительно доли в корпусе, у кого упадёт?

**Предсказание:** …

In [ ]:
freq = counts.float() / counts.sum()
p_neg = negative_distribution(counts)
rows = top[:5].tolist() + [word2id[w] for w in RARE.values()]
print(f"{'слово':>10} {'корпус':>8} {'P_neg':>8} {'P_neg/корпус':>13}")
for i in rows:
    print(f"{id2word[i]:>10} {freq[i].item():8.4f} {p_neg[i].item():8.4f} {(p_neg[i] / freq[i]).item():13.2f}")

## 3. Loss до обучения

🔮 **Предскажи:** `dim=16`, `K=5` — какое значение loss на случайном батче до первого шага и почему?

**Предсказание:** …

In [ ]:
DIM, WINDOW, K = 16, 2, 5
torch.manual_seed(0)
model = SGNS(V, DIM)
g = torch.Generator().manual_seed(0)
batch = pairs[torch.randperm(len(pairs), generator=g)[:512]]
neg = sample_negatives(p_neg, len(batch), K, g)
with torch.no_grad():
    print("loss до обучения:", model(batch[:, 0], batch[:, 1], neg).item())

## 4. Обучение

🔮 **Предскажи:** как будет выглядеть кривая loss за 8 эпох (`dim=16`, `window=2`, `K=5`, Adam `lr=0.01`)? До какого примерно значения упадёт и почему не до 0?

**Предсказание:** …

In [ ]:
losses = train(model, pairs, p_neg, epochs=8, batch_size=512, k=K, lr=0.01)
print([round(x, 3) for x in losses])
plt.plot(range(1, len(losses) + 1), losses, marker="o")
plt.xlabel("эпоха"); plt.ylabel("средний loss"); plt.title("SGNS, toy-корпус")
plt.grid(alpha=0.3); plt.show()

## 5. Ближайшие соседи

🔮 **Предскажи:** соседи для частого слова каждой темы (`кошка`, `хлеб`, `автобус`), для редкого (`ёж`) и для служебного `и`. Где выйдет хуже всего?

**Предсказание:** …

In [ ]:
for w in ["кошка", "хлеб", "автобус", "ёж", "и"]:
    print(f"{w:>8}:", ", ".join(f"{n} {s:.2f}" for n, s in most_similar(model, w, word2id, id2word, k=5)))

## 6. Двумерная картинка

🔮 **Предскажи:** PCA центральных векторов на плоскость — что будет видно? Где окажутся редкие слова и служебные?

**Предсказание:** …

In [ ]:
W = model.center.weight.detach()
U, S, Vt = torch.pca_lowrank(W, q=2)  # центрирует W внутри
xy = (W - W.mean(0)) @ Vt[:, :2]

topic_of = {w: t for t, ws in TOPICS.items() for w in ws} | {w: "служебные" for w in FUNCTION_WORDS}
colors = {"животные": "tab:green", "еда": "tab:orange", "транспорт": "tab:blue", "служебные": "tab:gray"}
plt.figure(figsize=(8, 6))
for t, c in colors.items():
    idx = [i for i, w in enumerate(id2word) if topic_of[w] == t]
    plt.scatter(xy[idx, 0], xy[idx, 1], c=c, label=t)
for i, w in enumerate(id2word):
    plt.annotate(w, xy[i].tolist(), fontsize=9, xytext=(3, 3), textcoords="offset points",
                 weight="bold" if w in RARE.values() else "normal")
plt.legend(); plt.title("PCA центральных векторов (редкие — жирным)"); plt.grid(alpha=0.3); plt.show()

## 7. ❓ Вопрос

Сравни редкое слово с частым из той же темы: соседи и место на картинке. Получилось ли оно хуже? Почему в реальном корпусе редкие слова обычно выходят хуже частых и что из теории это чинит?

**Ответ:** …

In [14]:
sentences = [["кот", "ест", "рыбу", "кот"], ["кот", "спит"], ["пёс", "ест", "мясо", "и", "кот", "ест"]]

uniq_word = list()
tf = {}

for doc in sentences:
    for word in doc:
        if word in tf:
            tf[word] += 1
        else:
            tf[word] = 1

for word, count in tf.items():
    if count < 1:
        pass
    else:
        uniq_word.append(word)
###
word2id = {}
id2word = []

for i, word in enumerate(uniq_word):
    word2id[word] = i
    id2word.append(word)
###
counts = []

for word in id2word:
    counts.append(tf[word])



print(f"tf: {tf}")
print(f"uniq_word: {uniq_word}")
print(f"word2id: {word2id}")
print(f"id2word: {id2word}")
print(f"counts: {counts}")

tf: {'кот': 4, 'ест': 3, 'рыбу': 1, 'спит': 1, 'пёс': 1, 'мясо': 1, 'и': 1}
uniq_word: ['кот', 'ест', 'рыбу', 'спит', 'пёс', 'мясо', 'и']
word2id: {'кот': 0, 'ест': 1, 'рыбу': 2, 'спит': 3, 'пёс': 4, 'мясо': 5, 'и': 6}
id2word: ['кот', 'ест', 'рыбу', 'спит', 'пёс', 'мясо', 'и']
counts: [4, 3, 1, 1, 1, 1, 1]


In [18]:
import torch

def build_vocab(
    sentences: list[list[str]], min_count: int = 1
):
    """Словарь по корпусу. Слова с частотой < min_count выкидываются.
    Возвращает:
        word2id: слово → id (0..V-1)
        id2word: список длины V, id2word[i] — слово с id i
        counts:  LongTensor (V,), counts[i] — сколько раз слово i встретилось в корпусе
    """
    ###
    uniq_word = []
    tf = {}

    for doc in sentences:
        for word in doc:
            if word in tf:
                tf[word] += 1
            else:
                tf[word] = 1

    for word, count in tf.items():
        if count < min_count:
            pass
        else:
            uniq_word.append(word)
    ###
    word2id = {}
    id2word = []

    for i, word in enumerate(uniq_word):
        word2id[word] = i
        id2word.append(word)
    ###
    counts = []

    for word in id2word:
        counts.append(tf[word])
    
    counts = torch.tensor(counts, dtype=torch.long)

    return word2id, id2word, counts


print(type(build_vocab([["кот", "ест", "рыбу", "кот"], ["кот", "спит"], ["пёс", "ест", "мясо", "и", "кот", "ест"]])))

<class 'tuple'>


In [ ]:
# s = "кот ест рыбу утром"
# ids = [7, 3, 9, 5]


def skipgram_pairs(ids: list[int], window: int) -> list[tuple[int, int]]:
    """Обучающие пары Skip-Gram для ОДНОГО предложения (уже переведённого в id).
    Для каждой позиции t: пары (ids[t], ids[t+j]) для всех 1 ≤ |j| ≤ window, не выходя за края.
    Порядок: по t слева направо, внутри t — по j от -window до +window.
    Возвращает список пар (center_id, context_id).
    """
    pairs = []

    for i in ids:
        

In [46]:
ids = [7, 3, 9, 5] # [(7,3), (3,7), (3,9), (9,3), (9,5), (5,9)]

window = 1
pairs = []

pairs.append(
    (7, 3))

print(pairs)

[(7, 3)]
